# Practical 02 — Symbolic Manipulation and Pattern Matching

**Artificial Intelligence 3(2-1) · BS Computer Science · Semester III**

| | |
|---|---|
| **Week** | 2 |
| **Related lectures** | Lectures 4, 18 and 19 |
| **Duration** | 3 hours |
| **Submission** | This notebook, run top to bottom from a clean kernel |

## 1. Objectives

By the end of this session you will be able to:

1. Represent statements as structured data rather than as plain strings.
2. Write a pattern matcher that recognises a template containing variables.
3. Build a transformation system that rewrites one expression into another.
4. Construct a conversational responder using pattern matching alone.
5. Explain what your program does and does not represent.

## 2. Background theory

Lecture 4 made the point that a machine manipulates symbols without any access to what they mean, and Lecture 19 showed two historical programs that exploited exactly this — producing responses that convinced people they were understood, using nothing but pattern matching and rearrangement.

Today you build the same machinery. A **pattern** is a template containing variables; **matching** is finding values for those variables that make the pattern identical to the input; and a **transformation** rewrites the captured pieces into a new expression. This is the informal ancestor of the unification procedure you will formalise in Practical 3.

**Key points to keep in mind while you work:**

- A pattern with a variable matches many inputs; the match returns what the variable captured.
- The program has no notion of what any symbol refers to — only its shape.
- Swapping person words ('my' to 'your') is what makes a reply read as though it engaged.
- If a fluent reply convinces a person, that says something about people, not about the program.

## 3. Apparatus and setup

**Software:** Python 3.10 or later. Standard library only — no external packages needed.

Nothing to install. Run the cell to confirm your kernel is working.

In [ ]:
print('Ready. No external packages required for this practical.')

## 4. Procedure

Work through these steps in order. The code here is given to you and should run as it stands — read it, run it, and make sure you understand what each part does before moving on to your own tasks in Section 5.

### Step 1 — Representing a statement as structure, not text

A string is opaque: you cannot easily ask what its parts are. Representing a statement as a list of tokens lets you inspect and rearrange it.

In [ ]:
sentence = 'i am worried about my exams'
tokens = sentence.split()
print('as a string :', repr(sentence))
print('as tokens   :', tokens)
print('third token :', tokens[2])

### Step 2 — A matcher for patterns with one variable

The pattern below uses `?X` to mean 'capture everything here'. The matcher returns a dictionary of what each variable captured, or None if the pattern does not fit.

In [ ]:
def match(pattern, tokens):
    """Match a token list against a pattern. '?X' captures the rest.
    Returns a dict of bindings, or None if there is no match."""
    bindings = {}
    for i, p in enumerate(pattern):
        if p.startswith('?'):
            bindings[p] = tokens[i:]      # capture everything remaining
            return bindings
        if i >= len(tokens) or p != tokens[i]:
            return None                   # a constant word did not match
    return bindings if len(pattern) == len(tokens) else None

print(match(['i', 'am', '?X'], 'i am worried about my exams'.split()))
print(match(['i', 'am', '?X'], 'she is worried'.split()))

### Step 3 — Transforming what was captured

Once you have captured a phrase, you can rewrite it. Swapping first-person for second-person words is what makes the reply read naturally.

In [ ]:
SWAP = {'my': 'your', 'i': 'you', 'me': 'you', 'am': 'are', 'mine': 'yours'}

def swap_person(tokens):
    return [SWAP.get(t, t) for t in tokens]

captured = 'worried about my exams'.split()
print('captured:', captured)
print('swapped :', swap_person(captured))

### Step 4 — Putting it together into a responder

A rule is a pattern plus a reply template. The responder tries each rule in turn and falls back on a generic prompt when nothing matches.

In [ ]:
RULES = [
    (['i', 'am', '?X'],      'How long have you been {X}?'),
    (['i', 'need', '?X'],    'Why do you need {X}?'),
    (['i', 'cannot', '?X'],  'What stops you being able to {X}?'),
]
FALLBACK = 'Tell me more about that.'

def respond(sentence):
    tokens = sentence.lower().split()
    for pattern, template in RULES:
        b = match(pattern, tokens)
        if b is not None:
            captured = swap_person(b['?X'])
            return template.format(X=' '.join(captured))
    return FALLBACK

for s in ['I am worried about my exams',
          'I need more time',
          'The weather is fine today']:
    print('%-32s -> %s' % (s, respond(s)))

## 5. Tasks

These are your work. Each cell marked `# TODO` is for you to complete. Write your own code; do not copy the procedure cells unchanged.

### Task 1 — Add five rules of your own

Extend the RULES table with at least five new patterns. At least two of them must capture a phrase and reuse it in the reply.

> **Marks:** 4

In [ ]:
RULES_EXTENDED = [
    (['i', 'like', '?X'], 'What do you like about {X}?'),
    (['i', 'want', '?X'], 'Why do you want {X}?'),
    (['i', 'need', '?X'], 'Why do you need {X}?'),
    (['i', 'have', '?X'], 'How does having {X} affect you?'),
    (['i', 'feel', '?X'], 'What makes you feel {X}?')
]

### Task 2 — Handle a variable in the middle of a pattern

The matcher above only captures at the end of a pattern. Extend it so a pattern such as `['i', '?X', 'about', 'exams']` works — the variable captures the words between two fixed anchors.

> **Marks:** 5

In [ ]:
def match_middle(pattern, tokens):
    index = pattern.index('?X')
    before = pattern[:index]
    after = pattern[index + 1:]
    if tokens[:len(before)] != before:
        return None
    if tokens[-len(after):] != after:
        return None
    captured = tokens[len(before):len(tokens) - len(after)]
    return {'?X': captured}

# Example
pattern = ['i', '?X', 'about', 'exams']
tokens = ['i', 'am', 'very', 'worried', 'about', 'exams']
print(match_middle(pattern, tokens))

### Task 3 — Build a simple expression simplifier

Lecture 20 described simplification by rewrite rules. Represent the expression `1 * x + 0` as a nested list and write rules that reduce it to `x`. Apply the rules repeatedly until nothing changes.

> **Marks:** 5

In [ ]:
expression = ['+', ['*', 1, 'x'], 0]
def simplify(expr):
    if expr == ['*', 1, 'x']:
        return 'x'
    if expr == ['+', 'x', 0]:
        return 'x'
    return expr
while expression != 'x':
    if expression == ['+', ['*', 1, 'x'], 0]:
        expression = ['+', 'x', 0]
    else:
        expression = simplify(expression)
print(expression)

### Task 4 — Test it on a classmate

Have someone else use your responder for a few exchanges without telling them how it works. Record what they assumed the program knew, in the observations table.

> **Marks:** 2

In [ ]:
What the classmate assumed:
1. The program understood what they were saying.
2. The program understood their feelings.
3. The program knew the meaning of their sentence.

What the program actually knew:
1. It only matched words with patterns.
2. It captured words and used them in a reply.
3. It followed predefined rules and did not understand the real meaning.

### Task 5 — State what your program represents

In a markdown cell, list exactly what your program stores. Then list what a user typically assumes it stores. Be specific and honest about the gap.

> **Marks:** 4

In [ ]:
### What my program stores
- The words entered by the user.
- Pattern rules. 
- Variables such as '?X'.
- The words captured by '?X'.
- Reply templates.
- Rules for changing words like `my` to `your`.

### What a user assumes it stores
- The meaning of the user's sentence.
- The user's feelings.
- The user's actual situation.
- Knowledge about what the user is talking about.

### The gap
- The program does not really understand the user's sentence or feelings. 
- It only matches patterns, captures words, and generates a reply using predefined rules.

## 6. Observations

Complete this table from your own results. Do not leave it blank — the observations carry marks and are what the viva is based on.

| Input sentence | Rule that matched | Reply produced | Sensible? |
|---|---|---|---|
| I am worried about my exams | ['i','am','?X] | How long have you been worried about your exams? | Yes |
| I need more time | ['i','need','?X] | Why do you need more time? | Yes |
| I like computer science | ['i','like','?X] | What do you like about computer science? | Yes |

**Tester's reaction (Task 4)**

| What they assumed the program knew | What it actually stores |
|---|---|
| It understood what they were saying. | It stores words, patterns, and reply rules. |

## 7. Discussion

Answer in your own words, in the cell below. Two or three sentences each is enough.

1. Your responder produced sensible replies. What, if anything, does it understand? Justify your answer by reference to what is actually stored.
2. Why does swapping person words make such a large difference to how the reply reads?
3. Lecture 19 noted that people who knew how such a program worked still felt understood by it. What does that tell you about evaluating conversational systems today?

*Your answers:*

1. The responder does not really understand the user's meaning. It only stores words, patterns, captured phrases and reply templates, and uses them to make a response.
2. Swapping person words makes the reply sound more natural and personal. For example, changing "my" to "your" makes the response look like it is talking directly to the user.
3. It shows that people can feel understood even when a program is only using simple rules. So, when evaluating conversational systems, we should look at how they actually work, not only how their replies sound.


## 8. Viva questions

Be ready to answer these on the spot. They test whether you understand what you ran, not whether you can recite the code.

1. Show me where in your code the program decides what a sentence means. (There is no such place — can you say why?)
2. What does your matcher return when the pattern fits, and what when it does not?
3. Why did we represent the sentence as tokens rather than as a string?
4. What happens to your responder when no rule matches, and is that a good design?
5. How is your matcher related to the unification you will write next week?
6. Your tester assumed the program knew something. What, and why did they assume it?

## 9. Submission checklist

- [ ] Notebook runs top to bottom from a clean kernel with no errors
- [ ] Every `# TODO` cell is completed with your own code
- [ ] The observations table is filled in with your actual results
- [ ] Discussion answers are written in your own words
- [ ] Your name and roll number are in the cell below
- [ ] File named `P02_<your-roll-number>.ipynb`

In [ ]:
NAME = "Umme Rubab"
ROLL_NUMBER = "2025-cse-033"
print(NAME , ROLL_NUMBER)